# 06 - Métodos de Gauss
Vamos aprender sobre como usar os métodos de Gauss para resolução de sistemas lineares quadráticos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana6
```

Faça o checkout nessa nova branch:

```bash
git checkout semana6
```

<hr />

## Atividade 1
Escreva os métodos de gauss dentro do arquivo $utils/algoritmos.py$.

In [1]:
import numpy as np


def lu_pivot(A: np.ndarray):
    A = A.astype(float).copy()
    n = A.shape[0]
    P = np.eye(n)
    L = np.zeros((n, n))
    U = A.copy()

    for k in range(n):
        pivot = np.argmax(np.abs(U[k:, k])) + k
        if np.isclose(U[pivot, k], 0.0):
            raise np.linalg.LinAlgError("Matriz singular ou quase singular.")
        if pivot != k:
            U[[k, pivot], k:] = U[[pivot, k], k:]
            P[[k, pivot], :] = P[[pivot, k], :]
            if k > 0:
                L[[k, pivot], :k] = L[[pivot, k], :k]

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    np.fill_diagonal(L, 1.0)
    return P, L, U


def lb(L: np.ndarray, B: np.ndarray) -> np.ndarray:
    n = L.shape[0]
    Y = np.zeros(n)
    for i in range(n):
        Y[i] = B[i] - np.dot(L[i, :i], Y[:i])
    return Y


def uy(U: np.ndarray, Y: np.ndarray) -> np.ndarray:
    n = U.shape[0]
    X = np.zeros(n)
    for i in reversed(range(n)):
        if np.isclose(U[i, i], 0.0):
            raise np.linalg.LinAlgError(
                "U possui pivô nulo; sistema sem solução única."
            )
        X[i] = (Y[i] - np.dot(U[i, i + 1 :], X[i + 1 :])) / U[i, i]
    return X


def lu(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    P, L, U = lu_pivot(A)
    Pb = P @ B
    Y = lb(L, Pb)
    X = uy(U, Y)
    return X


def jacobi(A: np.ndarray, B: np.ndarray, k: int, TOL: float) -> np.ndarray:
    A = A.astype(float)
    B = B.astype(float)
    n = B.shape[0]
    X = np.zeros(n)
    Xk = np.zeros(n)

    D = np.diag(A)
    if np.any(D == 0):
        raise ValueError("A possui elementos diagonais nulos; Jacobi pode falhar.")

    R = A - np.diagflat(D)

    for _ in range(k):
        Xk = (B - R @ X) / D
        if np.linalg.norm(Xk - X, ord=2) < TOL:
            return Xk
        X = Xk.copy()
    return X


def seidel(A: np.ndarray, B: np.ndarray, k: int, TOL: float) -> np.ndarray:
    A = A.astype(float)
    B = B.astype(float)
    n = B.shape[0]
    X = np.zeros(n)

    for _ in range(k):
        Xk = X.copy()
        for i in range(n):
            s1 = np.dot(A[i, :i], X[:i])
            s2 = np.dot(A[i, i + 1 :], Xk[i + 1 :])
            X[i] = (B[i] - s1 - s2) / A[i, i]
        if np.linalg.norm(X - Xk, ord=2) < TOL:
            return X
    return X

Consideramos o seguinte sistema linear:

$$
\begin{cases}
x_1 + x_2 + x_3 = 1\\
4x_1 + 4x_2 + 2x_3 = 2\\
2x_1 + x_2 - x_3 = 0
\end{cases}
$$

Na sua forma matricial, este sistema é escrito como

$$
Ax = b \iff
\begin{bmatrix}
1 & 1 & 1 \\
4 & 4 & 2 \\
2 & 1 & -1
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix}
=
\begin{bmatrix}
1 \\
2 \\
0
\end{bmatrix}
$$

Podemos resolver em Python usando a biblioteca NumPy.

In [1]:
import numpy as np

def main():
    A = np.array([
        [1, 1,  1],
        [4, 4,  2],
        [2, 1, -1]
    ], dtype=float)
    B = np.array([1, 2, 0], dtype=float)
    print("Matriz A:")
    print(A)
    print("\nVetor B:")
    print(B)
    print("\nSolução NumPy x:")
    X = np.linalg.solve(A, B)
    print(X)

if __name__ == "__main__":
    main()

Matriz A:
[[ 1.  1.  1.]
 [ 4.  4.  2.]
 [ 2.  1. -1.]]

Vetor B:
[1. 2. 0.]

Solução NumPy x:
[ 1. -1.  1.]


Agora, resolva usando os métodos de gauss e compare os resultados.

In [1]:
import sys
import os

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import (
    lu,
    jacobi,
    seidel,
)

def main():
    A = np.array([[1, 1, 1], [4, 4, 2], [2, 1, -1]], dtype=float)
    B = np.array([1, 2, 0], dtype=float)
    print("Matriz A:")
    print(A)
    print("\nVetor B:")
    print(B)
    print("\nSolução NumPy x:")
    X = np.linalg.solve(A, B)
    print(X)
    print("\nSolução LU x:")
    X = lu(A, B)
    print(X)
    A = np.array([[2, 1, -1], [4, 4, 2], [1, 1, 1]], dtype=float)
    B = np.array([0, 2, 1], dtype=float)
    print("\nMatriz A:")
    print(A)
    print("\nVetor B:")
    print(B)
    print("\nSolução Jacobi x:")
    X = jacobi(A, B, 100, 1e-8)
    print(X)
    print("\nSolução Seidel x:")
    X = seidel(A, B, 100, 1e-8)
    print(X)


if __name__ == "__main__":
    main()

Matriz A:
[[ 1.  1.  1.]
 [ 4.  4.  2.]
 [ 2.  1. -1.]]

Vetor B:
[1. 2. 0.]

Solução NumPy x:
[ 1. -1.  1.]

Solução LU x:
[ 1. -1.  1.]

Matriz A:
[[ 2.  1. -1.]
 [ 4.  4.  2.]
 [ 1.  1.  1.]]

Vetor B:
[0. 2. 1.]

Solução Jacobi x:
[ 0.99999507 -0.99999329  0.99999799]

Solução Seidel x:
[ 1. -1.  1.]


## Atividade 2
Resolva o seguinte sistema pelo método de Jacobi e Gauss–Seidel:

$$
\begin{cases}
5x_1 + x_2 + x_3 = 50 \\
-x_1 + 3x_2 - x_3 = 10 \\
x_1 + 2x_2 + 10x_3 = -30
\end{cases}
$$

Use como critério de paragem tolerância inferior a $10^{-3}$  
e inicialize com $x^0 = y^0 = z^0 = 0$.

In [1]:
import numpy as np
from utils.algoritmos import jacobi, seidel

A_2 = np.array([[5, 1, 1], [-1, 3, -1], [1, 2, 10]], dtype=float)
b_2 = np.array([50, 10, -30], dtype=float)
reference_2 = np.linalg.solve(A_2, b_2)
print("Partindo de x⁽⁰⁾ = (0, 0, 0), com TOL = 10⁻⁶:")
for name, method in (("Jacobi", jacobi), ("Gauss–Seidel", seidel)):
    x = method(A_2, b_2, 1000, 1e-6)
    residual = np.linalg.norm(A_2 @ x - b_2, ord=np.inf)
    print(f"{name}: {x}, resíduo máximo = {residual:.2e}")
    assert residual < 1e-3
print(f"Conferência com NumPy: {reference_2}")


Partindo de x⁽⁰⁾ = (0, 0, 0), com TOL = 10⁻⁶:
Jacobi: [ 9.99999986  5.00000014 -5.00000011], resíduo máximo = 9.47e-07
Gauss–Seidel: [ 9.99999996  4.99999996 -4.99999999], resíduo máximo = 2.37e-07
Conferência com NumPy: [10.  5. -5.]


## Atividade 3
Faça uma permutação de linhas no sistema abaixo e resolva pelos métodos de Jacobi e Gauss–Seidel:

$$
\begin{cases}
x_{1} + 10x_{2} + 3x_{3} = 27 \\
4x_{1} + x_{3} = 6 \\
2x_{1} + x_{2} + 4x_{3} = 12
\end{cases}
$$

In [1]:
import numpy as np
from utils.algoritmos import jacobi, seidel

A_3 = np.array([[1, 10, 3], [4, 0, 1], [2, 1, 4]], dtype=float)
b_3 = np.array([27, 6, 12], dtype=float)
# As equações 2, 1 e 3 dão a diagonal 4, 10 e 4, respectivamente.
order = [1, 0, 2]
reordered_A, reordered_b = A_3[order, :], b_3[order]
print("Matriz após permutar as duas primeiras linhas:\n", reordered_A)
for name, method in (("Jacobi", jacobi), ("Gauss–Seidel", seidel)):
    x = method(reordered_A, reordered_b, 1000, 1e-8)
    residual = np.linalg.norm(A_3 @ x - b_3, ord=np.inf)
    print(f"{name}: {x}; resíduo no sistema original = {residual:.2e}")
    assert residual < 1e-6
print("Conferência com NumPy:", np.linalg.solve(A_3, b_3))


Matriz após permutar as duas primeiras linhas:
 [[ 4.  0.  1.]
 [ 1. 10.  3.]
 [ 2.  1.  4.]]
Jacobi: [1. 2. 2.]; resíduo no sistema original = 2.06e-08
Gauss–Seidel: [1. 2. 2.]; resíduo no sistema original = 1.35e-08
Conferência com NumPy: [1. 2. 2.]


## Atividade 4
O circuito linear da Figura pode ser modelado pelo sistema dado a seguir.  
Escreva esse sistema na forma matricial sendo as tensões $V_1, V_2, V_3, V_4, V_5$ as cinco incógnitas.  
Resolva esse problema quando $V = 127$ e:

- $R_1 = R_2 = R_3 = R_4 = 2$, $R_5 = R_6 = R_7 = 100$ e $R_8 = 50$
- $R_1 = R_2 = R_3 = R_4 = 2$, $R_5 = 50$, $R_6 = R_7 = R_8 = 100$

Sistema de equações:

$$
V_1 = V
$$

$$
\frac{V_1 - V_2}{R_1} + \frac{V_5 - V_2}{R_2} - \frac{V_2}{R_5} = 0
$$

$$
\frac{V_2 - V_3}{R_2} + \frac{V_4 - V_3}{R_3} - \frac{V_3}{R_6} = 0
$$

$$
\frac{V_3 - V_4}{R_3} + \frac{V_5 - V_4}{R_4} - \frac{V_4}{R_7} = 0
$$

$$
\frac{V_4 - V_5}{R_4} - \frac{V_5}{R_8} = 0
$$

![Circuito](https://www.ufrgs.br/reamat/CalculoNumerico/livro-py/main11x.png)

Complete a tabela abaixo representando a solução com **4 algarismos significativos**:

| Caso | $V_1$ | $V_2$ | $V_3$ | $V_4$ | $V_5$ |
|------|-------|-------|-------|-------|-------|
| a    | 127.0 | 108.1 | 100.5 | 94.96 | 91.31 |
| b    | 127.0 | 110.6 | 104.5 | 100.5 | 98.53 |

Então, refaça este problema reduzindo o sistema para apenas 4 incógnitas ($V_2, V_3, V_4, V_5$).

In [1]:
import numpy as np
from utils.algoritmos import lu, jacobi, seidel

def circuit_system(resistors, voltage=127.0):
    r1, r2, r3, r4, r5, r6, r7, r8 = map(float, resistors)
    # A matriz corresponde, na ordem, às cinco equações do enunciado.
    A = np.array([
        [1, 0, 0, 0, 0],
        [-1/r1, 1/r1 + 1/r2 + 1/r5, 0, 0, -1/r2],
        [0, -1/r2, 1/r2 + 1/r3 + 1/r6, -1/r3, 0],
        [0, 0, -1/r3, 1/r3 + 1/r4 + 1/r7, -1/r4],
        [0, 0, 0, -1/r4, 1/r4 + 1/r8],
    ], dtype=float)
    b = np.array([voltage, 0, 0, 0, 0], dtype=float)
    return A, b

for case, resistors in (
    ("a", (2, 2, 2, 2, 100, 100, 100, 50)),
    ("b", (2, 2, 2, 2, 50, 100, 100, 100)),
):
    A, b = circuit_system(resistors)
    # V1 = V já é conhecido: Ared é o bloco de V2 a V5.
    Ared = A[1:, 1:]
    bred = b[1:] - A[1:, 0] * b[0]
    full = lu(A, b)
    reduced = lu(Ared, bred)
    jac = jacobi(Ared, bred, 10000, 1e-10)
    sei = seidel(Ared, bred, 10000, 1e-10)
    print(f"Caso {case}: sistema completo Ax=b com A=\n{A}\nb={b}")
    print(f"Caso {case}: sistema reduzido A₄x₄=b₄ com A₄=\n{Ared}\nb₄={bred}")
    print("V1 a V5, 4 algarismos significativos:",
          ", ".join(f"{value:#.4g}" for value in full))
    print("Resíduo máximo de LU (5x5):", np.linalg.norm(A @ full - b, ord=np.inf))
    assert np.allclose(full[1:], reduced, atol=1e-9)
    assert np.allclose(reduced, jac, atol=1e-6)
    assert np.allclose(reduced, sei, atol=1e-6)
    assert np.allclose(full, np.linalg.solve(A, b), atol=1e-9)


Caso a: sistema completo Ax=b com A=
[[ 1.    0.    0.    0.    0.  ]
 [-0.5   1.01  0.    0.   -0.5 ]
 [ 0.   -0.5   1.01 -0.5   0.  ]
 [ 0.    0.   -0.5   1.01 -0.5 ]
 [ 0.    0.    0.   -0.5   0.52]]
b=[127.   0.   0.   0.   0.]
Caso a: sistema reduzido A₄x₄=b₄ com A₄=
[[ 1.01  0.    0.   -0.5 ]
 [-0.5   1.01 -0.5   0.  ]
 [ 0.   -0.5   1.01 -0.5 ]
 [ 0.    0.   -0.5   0.52]]
b₄=[63.5  0.   0.   0. ]
V1 a V5, 4 algarismos significativos: 127.0, 108.1, 100.5, 94.96, 91.31
Resíduo máximo de LU (5x5): 2.1316282072803006e-14
Caso b: sistema completo Ax=b com A=
[[ 1.    0.    0.    0.    0.  ]
 [-0.5   1.02  0.    0.   -0.5 ]
 [ 0.   -0.5   1.01 -0.5   0.  ]
 [ 0.    0.   -0.5   1.01 -0.5 ]
 [ 0.    0.    0.   -0.5   0.51]]
b=[127.   0.   0.   0.   0.]
Caso b: sistema reduzido A₄x₄=b₄ com A₄=
[[ 1.02  0.    0.   -0.5 ]
 [-0.5   1.01 -0.5   0.  ]
 [ 0.   -0.5   1.01 -0.5 ]
 [ 0.    0.   -0.5   0.51]]
b₄=[63.5  0.   0.   0. ]
V1 a V5, 4 algarismos significativos: 127.0, 110.6, 104.5, 100.

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 6"
git push origin semana6
```